# Corporate Signal Intelligence  
## Data Analysis & Cleansing Notebook

> This notebook documents the transition from **raw collected data** to **clean, validated, model-ready datasets** for the Corporate Signal Intelligence project.

---

## Project Stage

The data collection phase has already produced two main analytical layers:

| Layer | Source | Purpose |
|---|---|---|
| **Market Data Layer** | Stooq | Historical daily price and volume data |
| **Corporate Data Layer** | SEC EDGAR | Company filings, disclosures, and financial facts |

This notebook focuses on validating, cleaning, and preparing these datasets before feature engineering and machine learning.

---

## Data Sources

### Market Data — Stooq

The market dataset was collected from the **Stooq CSV API** and contains historical daily trading data for the monitored companies.

```text
ticker
date
open
high
low
close
volume
source
collected_at
```

This dataset will support daily returns, price variation, volume variation, rolling volatility, volume spikes, and market anomaly detection.

### Corporate Data — SEC EDGAR

The corporate dataset was collected from **SEC EDGAR**, using public company submissions and XBRL company facts.

```text
sec_companies_selected_df
sec_filings_df
sec_facts_df
```

The SEC datasets contain company metadata, filing activity, annual reports, quarterly reports, current reports, and structured financial concepts.

| Form | Meaning |
|---|---|
| **10-K** | Annual report |
| **10-Q** | Quarterly report |
| **8-K** | Current report / material corporate event |

---

## Cleansing Goals

This notebook will perform the following tasks:

1. Load raw collected datasets.
2. Inspect dataset shapes, columns, and date ranges.
3. Validate missing values and duplicated records.
4. Standardize date and numeric formats.
5. Filter relevant SEC forms and financial concepts.
6. Remove or flag invalid records.
7. Create clean intermediate datasets.
8. Prepare the data for feature engineering and anomaly detection.

---

## Expected Outputs

By the end of this notebook, the project should produce:

```text
clean_market_data
clean_company_metadata
clean_sec_filings
clean_sec_facts
```

These datasets will later be used to build market anomaly detection models, corporate risk scoring logic, financial signal monitoring, Groq-powered executive briefings, FastAPI endpoints, and dashboard visualizations.

---

## Final System Context

```text
Stooq Historical Market Data
        +
SEC EDGAR Corporate Disclosures
        +
Neon PostgreSQL
        +
FastAPI REST API
        +
Scikit-learn Anomaly Detection
        +
Groq Executive Briefings
        +
Dashboard Layer
```

This notebook represents the bridge between **raw data collection** and **machine learning-ready analytical datasets**.

In [1]:
# Installing Dependencies

%pip install missingno openpyxl pyarrow

print("\nDependencies installed successfully!")

Note: you may need to restart the kernel to use updated packages.

Dependencies installed successfully!


In [5]:
# Importing libraries

import os
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from pathlib import Path

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [3]:
# Checking the enviroment

import sys

print(sys.executable)
print(sys.version)

/home/a1rm4x/Documents/GitHub/corporate-signal-intelligence/.venv/bin/python
3.14.4 (main, Apr  8 2026, 17:48:49) [GCC 15.2.1 20260209]


In [6]:
DATA_DIR = Path("data")

market_raw_path = DATA_DIR / "stooq_market_raw.csv"
companies_path = DATA_DIR / "sec_companies_selected.csv"
filings_raw_path = DATA_DIR / "sec_filings_raw.csv"
important_filings_path = DATA_DIR / "sec_important_filings.csv"
facts_raw_path = DATA_DIR / "sec_company_facts_raw.csv"

print("Market data:", market_raw_path.exists(), market_raw_path)
print("Companies:", companies_path.exists(), companies_path)
print("SEC filings:", filings_raw_path.exists(), filings_raw_path)
print("Important filings:", important_filings_path.exists(), important_filings_path)
print("SEC facts:", facts_raw_path.exists(), facts_raw_path)

Market data: True data/stooq_market_raw.csv
Companies: True data/sec_companies_selected.csv
SEC filings: True data/sec_filings_raw.csv
Important filings: True data/sec_important_filings.csv
SEC facts: True data/sec_company_facts_raw.csv


In [7]:
# Checking the dataframe shape

market_raw_df = pd.read_csv(market_raw_path)
companies_df = pd.read_csv(companies_path)
sec_filings_raw_df = pd.read_csv(filings_raw_path)
sec_important_filings_df = pd.read_csv(important_filings_path)
sec_facts_raw_df = pd.read_csv(facts_raw_path)

print("market_raw_df:", market_raw_df.shape)
print("companies_df:", companies_df.shape)
print("sec_filings_raw_df:", sec_filings_raw_df.shape)
print("sec_important_filings_df:", sec_important_filings_df.shape)
print("sec_facts_raw_df:", sec_facts_raw_df.shape)

market_raw_df: (81996, 9)
companies_df: (10, 5)
sec_filings_raw_df: (10022, 10)
sec_important_filings_df: (1092, 10)
sec_facts_raw_df: (14797, 16)


In [9]:
# Verifying if we have al the expected tickers

EXPECTED_TICKERS = [
    "AAPL",
    "MSFT",
    "NVDA",
    "GOOGL",
    "AMZN",
    "META",
    "TSLA",
    "AMD",
    "INTC",
    "ORCL",
]

market_tickers = sorted(market_raw_df["ticker"].unique().tolist())
company_tickers = sorted(companies_df["ticker"].unique().tolist())

print("Expected tickers:", sorted(EXPECTED_TICKERS))
print("Market tickers:", market_tickers)
print("Company tickers:", company_tickers)

missing_in_market = sorted(set(EXPECTED_TICKERS) - set(market_tickers))
extra_in_market = sorted(set(market_tickers) - set(EXPECTED_TICKERS))

print("Missing in market_raw_df:", missing_in_market)
print("Extra in market_raw_df:", extra_in_market)

Expected tickers: ['AAPL', 'AMD', 'AMZN', 'GOOGL', 'INTC', 'META', 'MSFT', 'NVDA', 'ORCL', 'TSLA']
Market tickers: ['AAPL', 'AMD', 'AMZN', 'GOOGL', 'INTC', 'META', 'MSFT', 'NVDA', 'ORCL', 'TSLA']
Company tickers: ['AAPL', 'AMD', 'AMZN', 'GOOGL', 'INTC', 'META', 'MSFT', 'NVDA', 'ORCL', 'TSLA']
Missing in market_raw_df: []
Extra in market_raw_df: []


In [10]:
# Standardizing the data from 2015 forward

clean_market_df = market_raw_df[
    market_raw_df["date"] >= "2015-01-01"
].copy()

clean_market_df = clean_market_df.sort_values(["ticker", "date"]).reset_index(drop=True)

clean_market_df.groupby("ticker").agg(
    rows=("date", "count"),
    min_date=("date", "min"),
    max_date=("date", "max"),
)

,rows,min_date,max_date
ticker,,,
AAPL,2863,2015-01-02,2026-05-21
AMD,2863,2015-01-02,2026-05-21
AMZN,2863,2015-01-02,2026-05-21
GOOGL,2863,2015-01-02,2026-05-21
INTC,2863,2015-01-02,2026-05-21
META,2863,2015-01-02,2026-05-21
MSFT,2863,2015-01-02,2026-05-21
NVDA,2863,2015-01-02,2026-05-21
ORCL,2863,2015-01-02,2026-05-21


In [11]:
# Cleaning the companies metadata

clean_companies_df = companies_df.copy()

clean_companies_df["ticker"] = clean_companies_df["ticker"].str.upper().str.strip()
clean_companies_df["cik"] = clean_companies_df["cik"].astype(str).str.zfill(10)
clean_companies_df["company_name"] = clean_companies_df["company_name"].str.strip()
clean_companies_df["source"] = clean_companies_df["source"].str.strip()

clean_companies_df = clean_companies_df.drop_duplicates(subset=["ticker", "cik"])
clean_companies_df = clean_companies_df.sort_values("ticker").reset_index(drop=True)

clean_companies_df

,ticker,cik,company_name,source,collected_at
0,AAPL,0000320193,Apple Inc.,sec_edgar,2026-05-21 19:59:24.334377+00:00
1,AMD,0000002488,ADVANCED MICRO DEVICES INC,sec_edgar,2026-05-21 19:59:24.334385+00:00
2,AMZN,0001018724,AMAZON COM INC,sec_edgar,2026-05-21 19:59:24.334378+00:00
3,GOOGL,0001652044,Alphabet Inc.,sec_edgar,2026-05-21 19:59:24.334375+00:00
4,INTC,0000050863,INTEL CORP,sec_edgar,2026-05-21 19:59:24.334388+00:00
5,META,0001326801,"Meta Platforms, Inc.",sec_edgar,2026-05-21 19:59:24.334380+00:00
6,MSFT,0000789019,MICROSOFT CORP,sec_edgar,2026-05-21 19:59:24.334378+00:00
7,NVDA,0001045810,NVIDIA CORP,sec_edgar,2026-05-21 19:59:24.334370+00:00
8,ORCL,0001341439,ORACLE CORP,sec_edgar,2026-05-21 19:59:24.334389+00:00
9,TSLA,0001318605,"Tesla, Inc.",sec_edgar,2026-05-21 19:59:24.334381+00:00


In [12]:
# Standardizing and cleaning important fillings

clean_sec_filings_df = sec_important_filings_df.copy()

clean_sec_filings_df["ticker"] = clean_sec_filings_df["ticker"].str.upper().str.strip()
clean_sec_filings_df["cik"] = clean_sec_filings_df["cik"].astype(str).str.zfill(10)
clean_sec_filings_df["form_type"] = clean_sec_filings_df["form_type"].str.upper().str.strip()

clean_sec_filings_df["filing_date"] = pd.to_datetime(
    clean_sec_filings_df["filing_date"],
    errors="coerce"
)

clean_sec_filings_df["report_date"] = pd.to_datetime(
    clean_sec_filings_df["report_date"],
    errors="coerce"
)

clean_sec_filings_df["collected_at"] = pd.to_datetime(
    clean_sec_filings_df["collected_at"],
    errors="coerce"
)

clean_sec_filings_df = clean_sec_filings_df.drop_duplicates(
    subset=["ticker", "accession_number", "form_type"]
)

clean_sec_filings_df = clean_sec_filings_df.sort_values(
    ["ticker", "filing_date"],
    ascending=[True, False]
).reset_index(drop=True)

clean_sec_filings_df.head()

,ticker,cik,accession_number,filing_date,report_date,form_type,primary_document,filing_url,source,collected_at
0,AAPL,0000320193,0000320193-26-000013,2026-05-01,2026-03-28,10-Q,aapl-20260328.htm,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:10:02.868771+00:00
1,AAPL,0000320193,0000320193-26-000011,2026-04-30,2026-04-30,8-K,aapl-20260430.htm,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:10:02.868772+00:00
2,AAPL,0000320193,0001140361-26-015711,2026-04-20,2026-04-17,8-K,ef20071035_8k.htm,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:10:02.868776+00:00
3,AAPL,0000320193,0001140361-26-006577,2026-02-24,2026-02-24,8-K,ef20060722_8k.htm,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:10:02.868795+00:00
4,AAPL,0000320193,0000320193-26-000006,2026-01-30,2025-12-27,10-Q,aapl-20251227.htm,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:10:02.868804+00:00


In [13]:
# Cleaning the facts

clean_sec_facts_df = sec_facts_raw_df.copy()

clean_sec_facts_df["ticker"] = clean_sec_facts_df["ticker"].str.upper().str.strip()
clean_sec_facts_df["cik"] = clean_sec_facts_df["cik"].astype(str).str.zfill(10)
clean_sec_facts_df["concept"] = clean_sec_facts_df["concept"].str.strip()
clean_sec_facts_df["unit"] = clean_sec_facts_df["unit"].str.strip()
clean_sec_facts_df["form_type"] = clean_sec_facts_df["form_type"].str.upper().str.strip()

date_columns = ["start_date", "end_date", "filing_date", "collected_at"]

for col in date_columns:
    clean_sec_facts_df[col] = pd.to_datetime(clean_sec_facts_df[col], errors="coerce")

clean_sec_facts_df["value"] = pd.to_numeric(clean_sec_facts_df["value"], errors="coerce")

clean_sec_facts_df["reference_date"] = clean_sec_facts_df["end_date"]

clean_sec_facts_df["period_type"] = np.where(
    clean_sec_facts_df["start_date"].isna(),
    "instant",
    "duration"
)

clean_sec_facts_df = clean_sec_facts_df.dropna(
    subset=["ticker", "concept", "value", "end_date"]
)

clean_sec_facts_df = clean_sec_facts_df.drop_duplicates(
    subset=[
        "ticker",
        "concept",
        "unit",
        "value",
        "start_date",
        "end_date",
        "filing_date",
        "form_type",
        "fiscal_year",
        "fiscal_period",
    ]
)

clean_sec_facts_df = clean_sec_facts_df.sort_values(
    ["ticker", "concept", "end_date", "filing_date"],
    ascending=[True, True, False, False]
).reset_index(drop=True)

clean_sec_facts_df.head()

,ticker,cik,taxonomy,concept,label,description,unit,value,start_date,end_date,filing_date,form_type,fiscal_year,fiscal_period,source,collected_at,reference_date,period_type
0,AAPL,0000320193,us-gaap,Assets,Assets,Sum of the carrying amounts as of the balance ...,USD,371082000000,NaT,2026-03-28,2026-05-01,10-Q,"2,026.0000",Q2,sec_edgar,2026-05-21 20:10:03.821984+00:00,2026-03-28,instant
1,AAPL,0000320193,us-gaap,Assets,Assets,Sum of the carrying amounts as of the balance ...,USD,379297000000,NaT,2025-12-27,2026-01-30,10-Q,"2,026.0000",Q1,sec_edgar,2026-05-21 20:10:03.821984+00:00,2025-12-27,instant
2,AAPL,0000320193,us-gaap,Assets,Assets,Sum of the carrying amounts as of the balance ...,USD,359241000000,NaT,2025-09-27,2026-05-01,10-Q,"2,026.0000",Q2,sec_edgar,2026-05-21 20:10:03.821983+00:00,2025-09-27,instant
3,AAPL,0000320193,us-gaap,Assets,Assets,Sum of the carrying amounts as of the balance ...,USD,359241000000,NaT,2025-09-27,2026-01-30,10-Q,"2,026.0000",Q1,sec_edgar,2026-05-21 20:10:03.821982+00:00,2025-09-27,instant
4,AAPL,0000320193,us-gaap,Assets,Assets,Sum of the carrying amounts as of the balance ...,USD,359241000000,NaT,2025-09-27,2025-10-31,10-K,"2,025.0000",FY,sec_edgar,2026-05-21 20:10:03.821981+00:00,2025-09-27,instant


In [14]:
# Clean dataframe validation summary

clean_datasets = {
    "clean_market_df": clean_market_df,
    "clean_companies_df": clean_companies_df,
    "clean_sec_filings_df": clean_sec_filings_df,
    "clean_sec_facts_df": clean_sec_facts_df,
}

clean_overview = []

for name, df in clean_datasets.items():
    clean_overview.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicated_rows": df.duplicated().sum(),
        "total_missing_values": df.isna().sum().sum(),
    })

clean_overview_df = pd.DataFrame(clean_overview)

clean_overview_df

,dataset,rows,columns,duplicated_rows,total_missing_values
0,clean_market_df,28630,9,0,0
1,clean_companies_df,10,5,0,0
2,clean_sec_filings_df,1092,10,0,0
3,clean_sec_facts_df,14797,18,0,6318


In [15]:
# Checking all the missing values in clean_sec_facts_sf

missing_summary = []

for name, df in clean_datasets.items():
    missing = df.isna().sum()
    missing_pct = (df.isna().mean() * 100).round(2)

    temp = pd.DataFrame({
        "dataset": name,
        "column": missing.index,
        "missing_values": missing.values,
        "missing_pct": missing_pct.values,
    })

    temp = temp[temp["missing_values"] > 0]

    missing_summary.append(temp)

missing_summary_df = pd.concat(missing_summary, ignore_index=True)

missing_summary_df.sort_values(
    ["dataset", "missing_values"],
    ascending=[True, False]
)

,dataset,column,missing_values,missing_pct
0,clean_sec_facts_df,start_date,6106,41.2700
1,clean_sec_facts_df,fiscal_year,106,0.7200
2,clean_sec_facts_df,fiscal_period,106,0.7200


## Missing Values Analysis

The cleaned datasets were inspected for missing values after the initial standardization process.

The market data, company metadata, and SEC filings datasets do not contain missing values after cleaning. The only dataset with relevant missing values is `clean_sec_facts_df`.

Most missing values are concentrated in the `start_date` column. This is expected in SEC XBRL data because some financial concepts represent values measured over a period, while others represent values at a specific point in time.

Concepts such as revenue, net income, operating income, and R&D expenses are usually duration-based and may contain both `start_date` and `end_date`.

Concepts such as assets, liabilities, stockholders' equity, and cash are instant-based and usually contain only `end_date`.

Because of this, missing `start_date` values are not considered a data quality issue. The pipeline will use `end_date` as the main reference date for financial facts.

A small number of records also contain missing `fiscal_year` and `fiscal_period` values. These records represent less than 1% of the SEC facts dataset and can be removed or ignored later if those fields are required for feature engineering.

In [17]:
# Validating facts

clean_sec_facts_df.groupby(["concept", "period_type"]).agg(
    rows=("value", "count"),
    missing_start_date=("start_date", lambda x: x.isna().sum()),
    missing_end_date=("end_date", lambda x: x.isna().sum()),
    min_reference_date=("reference_date", "min"),
    max_reference_date=("reference_date", "max"),
).sort_values(["concept", "rows"], ascending=[True, False])

,,rows,missing_start_date,missing_end_date,min_reference_date,max_reference_date
concept,period_type,,,,,
Assets,instant,1292,1292,0,2008-09-27,2026-04-26
CashAndCashEquivalentsAtCarryingValue,instant,2142,2142,0,2006-09-30,2026-04-26
Liabilities,instant,677,677,0,2008-09-27,2026-04-26
NetIncomeLoss,duration,2663,0,0,2007-09-29,2026-04-26
OperatingIncomeLoss,duration,2264,0,0,2007-09-29,2026-04-26
ResearchAndDevelopmentExpense,duration,1907,0,0,2007-09-29,2026-04-26
RevenueFromContractWithCustomerExcludingAssessedTax,duration,960,0,0,2016-06-30,2026-03-31
Revenues,duration,897,0,0,2007-09-30,2026-04-26
StockholdersEquity,instant,1995,1995,0,2006-09-30,2026-04-26


In [18]:
# Saving clean datasets into the data folder

clean_market_df.to_csv("data/clean_market_data.csv", index=False)
clean_companies_df.to_csv("data/clean_company_metadata.csv", index=False)
clean_sec_filings_df.to_csv("data/clean_sec_filings.csv", index=False)
clean_sec_facts_df.to_csv("data/clean_sec_facts.csv", index=False)

print("Clean datasets saved successfully.")

Clean datasets saved successfully.


# Data Analysis & Cleansing — Conclusion

The data analysis and cleansing stage was successfully completed for the **Corporate Signal Intelligence** project.

This notebook validated the raw datasets collected from the two primary data sources used in the project:

| Data Layer | Source | Status |
|---|---|---|
| **Market Data** | Stooq | Cleaned and validated |
| **Company Metadata** | SEC EDGAR | Cleaned and validated |
| **SEC Filings** | SEC EDGAR | Cleaned and validated |
| **SEC Financial Facts** | SEC EDGAR XBRL | Cleaned and validated |

The market dataset was standardized from **2015 onward**, creating a cleaner and more comparable historical base across the selected companies. This generated the `clean_market_df` dataset, which contains daily price and volume data with no duplicated records and no missing values.

The company metadata dataset was also standardized, ensuring consistent ticker formatting, CIK formatting, company names, source labels, and collection timestamps.

The SEC filings dataset was filtered to retain the most relevant corporate disclosure forms for the first version of the pipeline, especially:

```text
10-K
10-Q
8-K
```

These filings will later be used to create filing activity features, corporate event signals, and risk-related indicators.

The SEC financial facts dataset was cleaned and enriched with additional fields such as:

```text
reference_date
period_type
```

The missing values analysis confirmed that most missing values are concentrated in the `start_date` column. This is expected behavior in SEC XBRL data because some concepts are **duration-based**, while others are **instant-based**. Therefore, missing `start_date` values are not considered a data quality problem. The pipeline will use `end_date` as the main reference date for financial facts.

The final cleaned datasets produced in this notebook are:

```text
clean_market_df
clean_companies_df
clean_sec_filings_df
clean_sec_facts_df
```

These datasets are now ready for the next stage of the project: **feature engineering**.

The next notebook will transform the cleaned market and corporate datasets into model-ready features, including market volatility indicators, price and volume anomaly signals, filing activity metrics, financial ratios, and corporate risk indicators.

At this point, the project has moved from raw data collection to a clean analytical foundation suitable for machine learning, risk scoring, FastAPI integration, and Groq-powered executive briefings.